# Approfondir les cumuls

Ces techniques prolongent le parcours, mais ne sont pas des prérequis du TP de l’après-midi.

**Question métier :** Que signifie une courbe cumulée quand les ventes mensuelles baissent ?

Les données sont montées en lecture seule. Enregistrez vos figures dans `/workspace/resultats`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/workspace/data/sample_-_superstore.xls'
orders = pd.read_excel(DATA, sheet_name='Orders', engine='xlrd')
orders['Order Date'] = pd.to_datetime(orders['Order Date'])

## Notion 1 — Construire un cumul des ventes


### Étape 1 — Agréger les mois

Le cumul se calcule après avoir ordonné et totalisé les périodes.

In [ ]:
m = orders.set_index('Order Date').resample('MS')[['Sales','Profit']].sum()
m['Sales_cumulees'] = m['Sales'].cumsum()
print(m[['Sales','Sales_cumulees']].head())

### Étape 2 — Tracer la progression

La courbe représente le total accumulé depuis le début.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(m.index, m['Sales_cumulees'])
ax.set(title='Ventes cumulées', ylabel='Unités monétaires cumulées')
plt.show()

### Étape 3 — Interpréter

Le cumul continue généralement à progresser même si un mois isolé baisse, car il ajoute les ventes de la période au total précédent.

### Exercice 1 — Contrôler le cumul final

Comparez le cumul du dernier mois à la somme de Sales dans le fichier.

**À rendre :** Deux montants et une phrase sur le périmètre couvert.

In [ ]:
m['Sales_cumulees'] = m['Sales'].cumsum()
print(m['Sales_cumulees'].iloc[-1], orders['Sales'].sum())

### Mise en commun

Les montants sont égaux à l’arrondi près : le dernier cumul comprend toutes les ventes mensuelles.

## Notion 2 — Mesurer une évolution depuis une base fixe


### Étape 1 — Choisir la référence

Chaque montant mensuel est divisé par le montant de janvier 2023.

In [ ]:
evolution_base = (m['Sales'] / m['Sales'].iloc[0] - 1) * 100
print(evolution_base.head())

### Étape 2 — Représenter les écarts à la base

La ligne zéro correspond au mois de référence.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(m.index, evolution_base)
ax.axhline(0, color='black', linewidth=1)
ax.set(title='Évolution par rapport au premier mois', ylabel='Évolution (%)')
plt.show()

### Étape 3 — Interpréter

La base reste fixe. Cette mesure compare les montants mensuels entre eux ; elle ne représente pas le cumul des ventes.

### Exercice 2 — Changer l’année de référence

Comment les ventes évoluent-elles au cours de 2024 si janvier 2024 sert de référence ? Tracez l’évolution en pourcentage et interprétez la valeur de décembre.

**À rendre :** Une courbe sur les douze mois de 2024, la variation de décembre et une phrase qui nomme la référence.

In [ ]:
ventes_2024 = m.loc["2024", "Sales"]
evolution_2024 = (ventes_2024 / ventes_2024.iloc[0] - 1) * 100
print(evolution_2024)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evolution_2024.index, evolution_2024.values, color="#275c47")
ax.axhline(0, color="black", linewidth=1)
ax.set(title="Ventes de 2024, référence janvier 2024", ylabel="Évolution (%)")
plt.tight_layout()
plt.show()

### Mise en commun

Le point de janvier 2024 vaut 0 %. En décembre, les ventes sont environ 309,13 % au-dessus de celles de janvier 2024. La référence est le montant mensuel de janvier 2024, sans cumul des ventes.

## Notion 3 — Lisser une série avec rolling


### Étape 1 — Calculer une fenêtre mobile

Une fenêtre de trois observations calcule la moyenne des trois derniers mois.

In [ ]:
m['mobile_3'] = m['Sales'].rolling(3, min_periods=3).mean()
print(m[['Sales','mobile_3']].head())

### Étape 2 — Comparer les deux courbes

Le lissage atténue une partie des fluctuations observées.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(m.index, m['Sales'], alpha=.5, label='Ventes mensuelles')
ax.plot(m.index, m['mobile_3'], label='Moyenne mobile 3 mois')
ax.set(title='Ventes et moyenne mobile', ylabel='Unités monétaires')
ax.legend()
plt.show()

### Étape 3 — Interpréter

Les deux premières valeurs sont absentes avec une fenêtre complète de trois mois. La moyenne mobile décrit les observations passées ; elle n’est pas une prévision.

### Exercice 3 — Vérifier une moyenne mobile à la main

Calculez la moyenne des ventes de janvier à mars 2023 puis comparez-la à rolling(3).

**À rendre :** Le calcul manuel, la valeur calculée et le nombre d’observations de la fenêtre.

In [ ]:
print(m['Sales'].iloc[:3].mean())
print(m.loc['2023-03-01','mobile_3'])

### Mise en commun

Les deux valeurs coïncident : la moyenne de mars 2023 comprend janvier, février et mars.

## Notion 4 — Calculer un ratio sur des cumuls


### Étape 1 — Cumuler les deux mesures

Le profit cumulé et les ventes cumulées doivent couvrir les mêmes mois.

In [ ]:
m['Profit_cumule'] = m['Profit'].cumsum()
m['Sales_cumulees'] = m['Sales'].cumsum()
m['Taux_cumule'] = m['Profit_cumule'] / m['Sales_cumulees'] * 100
print(m[['Profit_cumule','Sales_cumulees','Taux_cumule']].tail())

### Étape 2 — Tracer le ratio

Le graphique montre un taux calculé sur les cumuls, pas une moyenne de taux mensuels.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(m.index, m['Taux_cumule'])
ax.set(title='Profit cumulé rapporté aux ventes cumulées', ylabel='Profit / ventes (%)')
plt.show()

### Étape 3 — Interpréter

Le ratio cumulé rapporte le profit accumulé aux ventes accumulées sur la même période. Chaque mois contribue selon son montant de ventes ; une moyenne de taux mensuels donne le même poids à tous les mois.

### Exercice 4 — Recommencer les cumuls en janvier 2024

Comment le ratio profit cumulé/ventes cumulées évolue-t-il pendant 2024 ? Tracez-le en recommençant les cumuls en janvier, puis comparez sa valeur de décembre à la moyenne des taux mensuels de 2024.

**À rendre :** Une courbe du ratio cumulé sur 2024, les deux taux de fin d’analyse et une explication de leur différence.

In [ ]:
bilan_2024 = m.loc["2024", ["Sales", "Profit"]].copy()
bilan_2024["Sales_cumulees"] = bilan_2024["Sales"].cumsum()
bilan_2024["Profit_cumule"] = bilan_2024["Profit"].cumsum()
bilan_2024["Taux_cumule"] = bilan_2024["Profit_cumule"] / bilan_2024["Sales_cumulees"] * 100
bilan_2024["Taux_mensuel"] = bilan_2024["Profit"] / bilan_2024["Sales"] * 100
print("Ratio cumulé en décembre :", bilan_2024["Taux_cumule"].iloc[-1])
print("Moyenne des taux mensuels :", bilan_2024["Taux_mensuel"].mean())
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bilan_2024.index, bilan_2024["Taux_cumule"], color="#275c47")
ax.set(title="Profit cumulé / ventes cumulées en 2024", ylabel="Ratio (%)")
plt.tight_layout()
plt.show()

### Mise en commun

Le ratio cumulé atteint environ 13,11 % en décembre 2024, contre 12,25 % pour la moyenne des taux mensuels. Les cumuls commencent en janvier 2024. Le premier calcul pondère les mois par leurs ventes, tandis que le second donne le même poids à chaque mois.